# 🌾 Seasonal Agriculture Performance Analysis

## 📑 Table of Contents

- [Project Introduction](#project-introduction)
- [Problem Statement](#problem-statement)
- [Objectives](#objectives)
- [Dataset Description](#dataset-description)
- [Data Cleaning](#data-cleaning--preprocessing)
- [Exploratory Data Analysis](#exploratory-data-analysis)
- [Seasonal Performance Analysis](#seasonal-performance-analysis)
- [Crop Analysis](#crop-analysis)
- [Regional Analysis](#regional-analysis)
- [Resource & Water Efficiency](#resource--water-efficiency)
- [Environmental Analysis](#environmental-analysis)
- [Economic Performance](#economic-performance)
- [Statistical Analysis](#statistical-analysis)
- [Overall Performance Ranking](#overall-performance-ranking)
- [Key Findings](#key-findings)
- [Recommendations](#recommendations)
- [Conclusion](#conclusion)
- [Final Project Outcome](#final-project-outcome)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully.")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
df = pd.read_csv('seasonal_agriculture_performance_dataset.csv')

print("Dataset loaded successfully!")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
rows, columns = df.shape

print("Number of Rows    :", rows)
print("Number of Columns :", columns)

In [ ]:
print("Dataset Columns:\n")

for i, column in enumerate(df.columns, 1):
    print(f"{i}. {column}")

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.describe(include='object').T

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include='object').columns.tolist()

print("Numerical Columns:")
print(numeric_columns)

print("\nCategorical Columns:")
print(categorical_columns)

In [ ]:
for col in categorical_columns:
    print(f"\n{'='*50}")
    print(f"{col}")
    print(f"{'='*50}")
    print("Unique values:", df[col].nunique())
    print(df[col].unique())

In [ ]:
season_counts = df['Season'].value_counts()

print(season_counts)

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(data=df, x='Season')

plt.title('Distribution of Agricultural Records by Season')
plt.xlabel('Season')
plt.ylabel('Number of Records')

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.countplot(data=df, x='Crop', order=df['Crop'].value_counts().index)

plt.title('Distribution of Agricultural Records by Crop')
plt.xlabel('Crop')
plt.ylabel('Number of Records')
plt.xticks(rotation=45)

plt.show()

In [ ]:
print("Number of States   :", df['State'].nunique())
print("Number of Districts:", df['District'].nunique())

In [ ]:
plt.figure(figsize=(12, 5))

sns.countplot(
    data=df,
    x='State',
    order=df['State'].value_counts().index
)

plt.title('Agricultural Records by State')
plt.xlabel('State')
plt.ylabel('Number of Records')
plt.xticks(rotation=45)

plt.show()

In [ ]:
missing_values = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
})

missing_values = missing_values[
    missing_values['Missing_Count'] > 0
].sort_values('Missing_Count', ascending=False)

missing_values

duplicate_count = df.duplicated().sum()

print("Duplicate Rows:", duplicate_count)

In [ ]:
print("Total Records :", len(df))
print("Unique Farm IDs:", df['Farm_ID'].nunique())

In [ ]:
print("===== DATA QUALITY SUMMARY =====")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Duplicate rows:", df.duplicated().sum())
print("Missing cells:", df.isnull().sum().sum())
print("Numerical columns:", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))

🌾Data Cleaning & Preparation

We know from Dataset Understanding that the dataset has 4,000 rows, 28 columns, no duplicate rows, and only 120 missing cells. Now we'll clean it without over-processing it.

In [ ]:
clean_df = df.copy()

print("Working copy created.")
print("Shape:", clean_df.shape)

In [ ]:
clean_df.columns = clean_df.columns.str.strip()

print(clean_df.columns.tolist())

In [ ]:
categorical_columns = clean_df.select_dtypes(include='object').columns

for col in categorical_columns:
    print(f"\n{col}")
    print(clean_df[col].value_counts(dropna=False))

In [ ]:
numeric_columns = clean_df.select_dtypes(include=np.number).columns

negative_values = {}

for col in numeric_columns:
    count = (clean_df[col] < 0).sum()
    if count > 0:
        negative_values[col] = count

negative_values

In [ ]:
clean_df[numeric_columns].describe().T

In [ ]:
percentage_columns = [
    col for col in clean_df.columns
    if '%' in col
]

print(percentage_columns)

In [ ]:
for col in percentage_columns:
    print(f"\n{col}")
    print("Minimum:", clean_df[col].min())
    print("Maximum:", clean_df[col].max())

In [ ]:
missing = pd.DataFrame({
    'Missing_Count': clean_df.isnull().sum(),
    'Missing_Percentage': (clean_df.isnull().sum() / len(clean_df)) * 100
})

missing = missing[missing['Missing_Count'] > 0]

missing

In [ ]:
clean_df['Rainfall_mm'] = (
    clean_df.groupby('Season')['Rainfall_mm']
    .transform(lambda x: x.fillna(x.median()))
)

In [ ]:
clean_df['Rainfall_mm'].isnull().sum()

In [ ]:
clean_df['Soil_Moisture_pct'] = (
    clean_df.groupby('Season')['Soil_Moisture_pct']
    .transform(lambda x: x.fillna(x.median()))
)

In [ ]:
clean_df['Soil_Moisture_pct'].isnull().sum()

In [ ]:
print("Missing Yield:", clean_df['Yield_Tonnes_Ha'].isnull().sum())

In [ ]:
yield_df = clean_df.dropna(subset=['Yield_Tonnes_Ha']).copy()

print("Original records:", len(clean_df))
print("Yield analysis records:", len(yield_df))

In [ ]:
remaining_missing = clean_df.isnull().sum()

remaining_missing = remaining_missing[
    remaining_missing > 0
].sort_values(ascending=False)

remaining_missing

In [ ]:
print("Duplicate rows after cleaning:", clean_df.duplicated().sum())

In [ ]:
outlier_summary = []

for col in numeric_columns:
    Q1 = clean_df[col].quantile(0.25)
    Q3 = clean_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((clean_df[col] < lower) | (clean_df[col] > upper)).sum()

    outlier_summary.append({
        'Column': col,
        'Outliers': outliers,
        'Outlier_Percentage': round((outliers / len(clean_df)) * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary.sort_values(
    'Outliers',
    ascending=False
)

In [ ]:
print("=" * 50)
print("FINAL DATA QUALITY CHECK")
print("=" * 50)

print("Rows:", clean_df.shape[0])
print("Columns:", clean_df.shape[1])
print("Duplicate rows:", clean_df.duplicated().sum())
print("Total missing cells:", clean_df.isnull().sum().sum())

print("\nMissing values by column:")
print(clean_df.isnull().sum()[clean_df.isnull().sum() > 0])

In [ ]:
clean_df.to_csv(
    'seasonal_agriculture_cleaned.csv',
    index=False
)

print("Clean dataset saved successfully.")

In [ ]:
print("Original dataset shape :", df.shape)
print("Clean dataset shape   :", clean_df.shape)
print("Yield dataset shape   :", yield_df.shape)

                RAW DATA
                   │
                   ▼
          4,000 × 28 dataset
                   │
                   ▼
         Remove duplicates
             0 found
                   │
                   ▼
        Validate data types
                   │
                   ▼
        Check invalid values
                   │
                   ▼
    ┌──────────────┴────────────┐
    ▼                           ▼
    Rainfall / Soil Moisture          Yield
    88 missing values               32 missing
    │                             │
    ▼                             ▼
    Season-specific median     Keep as missing for valid analysis
    └──────────────┬─────────────┘
                   ▼
              CLEAN DATA
              4,000 × 28
                   │
                   ▼
          Ready for EDA

🌾Exploratory Data Analysis (EDA)

EDA is performed to understand the distribution, variation,
relationships, and major patterns present in the cleaned
agricultural dataset.

In [ ]:
print("=" * 60)
print("PHASE 3 — EXPLORATORY DATA ANALYSIS")
print("=" * 60)

print("\nClean Dataset Shape:", clean_df.shape)
print("Yield Analysis Dataset Shape:", yield_df.shape)

print("\nColumns:")
print(clean_df.columns.tolist())

In [ ]:
categorical_cols = ['Season', 'Crop', 'State', 'District', 'Irrigation_Method']

for col in categorical_cols:
    if col in clean_df.columns:
        print(f"\n{col} — {clean_df[col].nunique()} unique values")
        print(clean_df[col].value_counts())

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(data=clean_df, x='Season')

plt.title('Agricultural Records by Season')
plt.xlabel('Season')
plt.ylabel('Number of Records')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(
    data=clean_df,
    x='Crop',
    order=clean_df['Crop'].value_counts().index
)

plt.title('Distribution of Agricultural Records by Crop')
plt.xlabel('Crop')
plt.ylabel('Number of Records')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(
    data=clean_df,
    x='State',
    order=clean_df['State'].value_counts().index
)

plt.title('Distribution of Agricultural Records by State')
plt.xlabel('State')
plt.ylabel('Number of Records')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
environmental_cols = [
    'Rainfall_mm',
    'Temperature_C',
    'Humidity_pct',
    'Soil_Moisture_pct',
    'Soil_pH',
    'Sunlight_Hours'
]

available_env = [col for col in environmental_cols if col in clean_df.columns]

for col in available_env:
    plt.figure(figsize=(8,4))

    sns.histplot(clean_df[col], kde=True)

    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

In [ ]:
resource_cols = [
    'Nitrogen_kg_ha',
    'Phosphorus_kg_ha',
    'Potassium_kg_ha',
    'Water_Used_m3'
]

available_resources = [
    col for col in resource_cols
    if col in clean_df.columns
]

for col in available_resources:
    plt.figure(figsize=(8,4))

    sns.histplot(clean_df[col], kde=True)

    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

In [ ]:
performance_cols = [
    'Yield_Tonnes_Ha',
    'Production_Tonnes',
    'Revenue_INR',
    'Profit_INR'
]

for col in performance_cols:
    if col in yield_df.columns:
        plt.figure(figsize=(8,4))

        sns.histplot(yield_df[col], kde=True)

        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
        plt.tight_layout()
        plt.show()

In [ ]:
season_summary = yield_df.groupby('Season').agg({
    'Yield_Tonnes_Ha': 'mean',
    'Production_Tonnes': 'mean',
    'Revenue_INR': 'mean',
    'Profit_INR': 'mean'
}).round(2)

season_summary

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=yield_df,
    x='Season',
    y='Yield_Tonnes_Ha'
)

plt.title('Yield Distribution Across Seasons')
plt.xlabel('Season')
plt.ylabel('Yield (Tonnes/Ha)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=yield_df,
    x='Season',
    y='Production_Tonnes'
)

plt.title('Production Distribution Across Seasons')
plt.xlabel('Season')
plt.ylabel('Production (Tonnes)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=yield_df,
    x='Season',
    y='Profit_INR'
)

plt.title('Profit Distribution Across Seasons')
plt.xlabel('Season')
plt.ylabel('Profit (INR)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
if 'Water_Efficiency_t_per_1000m3' in clean_df.columns:

    plt.figure(figsize=(8,5))

    sns.boxplot(
        data=clean_df,
        x='Season',
        y='Water_Efficiency_t_per_1000m3'
    )

    plt.title('Water Efficiency Across Seasons')
    plt.xlabel('Season')
    plt.ylabel('Water Efficiency')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=clean_df,
    x='Season',
    y='Rainfall_mm'
)

plt.title('Rainfall Variation Across Seasons')
plt.xlabel('Season')
plt.ylabel('Rainfall (mm)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=clean_df,
    x='Season',
    y='Avg_Temperature_C'
)

plt.title('Temperature Variation Across Seasons')
plt.xlabel('Season')
plt.ylabel('Temperature (°C)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=clean_df,
    x='Season',
    y='Soil_Moisture_pct'
)

plt.title('Soil Moisture Variation Across Seasons')
plt.xlabel('Season')
plt.ylabel('Soil Moisture (%)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
numeric_df = yield_df.select_dtypes(include=np.number)

correlation_matrix = numeric_df.corr()

plt.figure(figsize=(16,12))

sns.heatmap(
    correlation_matrix,
    cmap='coolwarm',
    center=0,
    linewidths=0.5
)

plt.title('Correlation Heatmap of Agricultural Variables')
plt.tight_layout()
plt.show()

In [ ]:
yield_corr = (
    correlation_matrix['Yield_Tonnes_Ha']
    .drop('Yield_Tonnes_Ha')
    .sort_values(key=abs, ascending=False)
)

print("Variables most strongly related to Yield:")
print(yield_corr)

In [ ]:
important_cols = [
    'Rainfall_mm',
    'Temperature_C',
    'Humidity_pct',
    'Soil_Moisture_pct',
    'Water_Used_m3',
    'Yield_Tonnes_Ha',
    'Revenue_INR',
    'Profit_INR'
]

available_cols = [
    col for col in important_cols
    if col in yield_df.columns
]

season_environment_summary = (
    yield_df
    .groupby('Season')[available_cols]
    .mean()
    .round(2)
)

season_environment_summary

In [ ]:
print("=" * 60)
print("KEY EDA STATISTICS")
print("=" * 60)

print("\nAverage Yield by Season:")
print(
    yield_df.groupby('Season')['Yield_Tonnes_Ha']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

print("\nAverage Profit by Season:")
print(
    yield_df.groupby('Season')['Profit_INR']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

print("\nAverage Rainfall by Season:")
print(
    clean_df.groupby('Season')['Rainfall_mm']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

print("\nAverage Water Usage by Season:")
print(
    clean_df.groupby('Season')['Water_Used_m3']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

In [ ]:
season_temperature = (
    clean_df.groupby('Season')['Avg_Temperature_C']
    .mean()
    .round(2)
)

print("Average Temperature by Season:")
print(season_temperature)

🌾CORE SEASONAL PERFORMANCE ANALYSIS

This phase analyzes agricultural performance across different
seasons using yield, production, revenue, profit, resource
usage, and efficiency indicators.

In [ ]:
season_performance = yield_df.groupby('Season').agg(
    Average_Yield=('Yield_Tonnes_Ha', 'mean'),
    Average_Production=('Production_Tonnes', 'mean'),
    Average_Revenue=('Revenue_INR', 'mean'),
    Average_Cost=('Total_Cost_INR', 'mean'),
    Average_Profit=('Profit_INR', 'mean'),
    Average_Water_Used=('Water_Used_m3', 'mean'),
    Average_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean')
).round(2)

season_performance

In [ ]:
yield_by_season = (
    yield_df.groupby('Season')['Yield_Tonnes_Ha']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

print("Average Yield by Season:")
print(yield_by_season)

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=yield_df,
    x='Season',
    y='Yield_Tonnes_Ha',
    estimator='mean',
    errorbar=None
)

plt.title('Average Agricultural Yield by Season')
plt.xlabel('Season')
plt.ylabel('Average Yield (Tonnes/Ha)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
production_by_season = (
    yield_df.groupby('Season')['Production_Tonnes']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

print("Average Production by Season:")
print(production_by_season)

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=yield_df,
    x='Season',
    y='Production_Tonnes',
    estimator='mean',
    errorbar=None
)

plt.title('Average Agricultural Production by Season')
plt.xlabel('Season')
plt.ylabel('Average Production (Tonnes)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
economic_summary = yield_df.groupby('Season').agg(
    Revenue=('Revenue_INR', 'mean'),
    Cost=('Total_Cost_INR', 'mean'),
    Profit=('Profit_INR', 'mean')
).round(2)

economic_summary

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=yield_df,
    x='Season',
    y='Profit_INR',
    estimator='mean',
    errorbar=None
)

plt.axhline(0, linewidth=1)

plt.title('Average Profit by Season')
plt.xlabel('Season')
plt.ylabel('Average Profit (INR)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
crop_season_yield = (
    yield_df
    .groupby(['Season', 'Crop'])['Yield_Tonnes_Ha']
    .mean()
    .reset_index()
)

crop_season_yield.head(20)

In [ ]:
crop_yield_pivot = crop_season_yield.pivot(
    index='Crop',
    columns='Season',
    values='Yield_Tonnes_Ha'
)

plt.figure(figsize=(10,8))

sns.heatmap(
    crop_yield_pivot,
    annot=True,
    fmt='.2f',
    cmap='YlGnBu'
)

plt.title('Average Crop Yield Across Seasons')
plt.xlabel('Season')
plt.ylabel('Crop')
plt.tight_layout()
plt.show()

In [ ]:
best_crop_each_season = (
    crop_season_yield
    .loc[
        crop_season_yield.groupby('Season')['Yield_Tonnes_Ha']
        .idxmax()
    ]
    .sort_values('Season')
)

print("Best Performing Crop in Each Season:")
print(best_crop_each_season)

In [ ]:
worst_crop_each_season = (
    crop_season_yield
    .loc[
        crop_season_yield.groupby('Season')['Yield_Tonnes_Ha']
        .idxmin()
    ]
    .sort_values('Season')
)

print("Lowest Performing Crop in Each Season:")
print(worst_crop_each_season)

In [ ]:
state_season_yield = (
    yield_df
    .groupby(['Season', 'State'])['Yield_Tonnes_Ha']
    .mean()
    .reset_index()
)

state_season_yield.head(20)

In [ ]:
best_state_each_season = (
    state_season_yield
    .loc[
        state_season_yield.groupby('Season')['Yield_Tonnes_Ha']
        .idxmax()
    ]
    .sort_values('Season')
)

print("Best Performing State in Each Season:")
print(best_state_each_season)

In [ ]:
resource_summary = clean_df.groupby('Season').agg(
    Fertilizer=('Fertilizer_kg_ha', 'mean'),
    Pesticide=('Pesticide_Litre_ha', 'mean'),
    Water=('Water_Used_m3', 'mean'),
    Nitrogen=('Nitrogen_kg_ha', 'mean'),
    Phosphorus=('Phosphorus_kg_ha', 'mean'),
    Potassium=('Potassium_kg_ha', 'mean')
).round(2)

resource_summary

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=clean_df,
    x='Season',
    y='Water_Used_m3',
    estimator='mean',
    errorbar=None
)

plt.title('Average Water Usage by Season')
plt.xlabel('Season')
plt.ylabel('Water Used (m³)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
water_efficiency = (
    clean_df.groupby('Season')['Water_Efficiency_t_per_1000m3']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

print("Average Water Efficiency by Season:")
print(water_efficiency)

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=clean_df,
    x='Season',
    y='Water_Efficiency_t_per_1000m3',
    estimator='mean',
    errorbar=None
)

plt.title('Average Water Efficiency by Season')
plt.xlabel('Season')
plt.ylabel('Water Efficiency (Tonnes / 1000 m³)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
environmental_summary = clean_df.groupby('Season').agg(
    Rainfall=('Rainfall_mm', 'mean'),
    Temperature=('Avg_Temperature_C', 'mean'),
    Humidity=('Humidity_pct', 'mean'),
    Soil_Moisture=('Soil_Moisture_pct', 'mean'),
    Sunlight=('Sunlight_Hours_Day', 'mean')
).round(2)

environmental_summary

In [ ]:
season_complete_summary = yield_df.groupby('Season').agg(
    Rainfall=('Rainfall_mm', 'mean'),
    Temperature=('Avg_Temperature_C', 'mean'),
    Humidity=('Humidity_pct', 'mean'),
    Soil_Moisture=('Soil_Moisture_pct', 'mean'),
    Sunlight=('Sunlight_Hours_Day', 'mean'),
    Water_Used=('Water_Used_m3', 'mean'),
    Yield=('Yield_Tonnes_Ha', 'mean'),
    Revenue=('Revenue_INR', 'mean'),
    Profit=('Profit_INR', 'mean'),
    Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean')
).round(2)

season_complete_summary

In [ ]:
ranking = season_performance.copy()

ranking['Yield_Rank'] = ranking['Average_Yield'].rank(
    ascending=False
)

ranking['Profit_Rank'] = ranking['Average_Profit'].rank(
    ascending=False
)

ranking['Water_Efficiency_Rank'] = ranking[
    'Average_Water_Efficiency'
].rank(ascending=False)

ranking.sort_values('Yield_Rank')

In [ ]:
score_df = season_performance[
    [
        'Average_Yield',
        'Average_Profit',
        'Average_Water_Efficiency'
    ]
].copy()

# Normalize each metric to 0–1
for col in score_df.columns:
    min_val = score_df[col].min()
    max_val = score_df[col].max()

    if max_val != min_val:
        score_df[col + '_Score'] = (
            (score_df[col] - min_val) /
            (max_val - min_val)
        )
    else:
        score_df[col + '_Score'] = 1

score_df['Overall_Score'] = (
    score_df['Average_Yield_Score'] +
    score_df['Average_Profit_Score'] +
    score_df['Average_Water_Efficiency_Score']
) / 3

score_df = score_df.sort_values(
    'Overall_Score',
    ascending=False
)

score_df.round(3)

In [ ]:
print("=" * 65)
print("PHASE 4 — SEASONAL PERFORMANCE SUMMARY")
print("=" * 65)

best_yield_season = yield_by_season.idxmax()
best_profit_season = economic_summary['Profit'].idxmax()
best_efficiency_season = water_efficiency.idxmax()

print(f"\nHighest Average Yield     : {best_yield_season}")
print(f"Highest Average Profit    : {best_profit_season}")
print(f"Highest Water Efficiency  : {best_efficiency_season}")

print("\nAverage Yield:")
print(yield_by_season)

print("\nAverage Profit:")
print(economic_summary['Profit'].sort_values(ascending=False))

print("\nAverage Water Efficiency:")
print(water_efficiency.sort_values(ascending=False))

print("\nOverall Seasonal Ranking:")
print(score_df['Overall_Score'].round(3))

🌾RELATIONSHIPS & STATISTICAL ANALYSIS

This phase investigates relationships between agricultural
performance, environmental conditions, resource usage, and
economic outcomes.

Statistical tests are used to determine whether observed
seasonal differences are statistically significant.

In [ ]:
yield_correlations = (
    yield_df.select_dtypes(include=np.number)
    .corr()['Yield_Tonnes_Ha']
    .drop('Yield_Tonnes_Ha')
    .sort_values(key=abs, ascending=False)
)

print("Correlation with Yield:")
print(yield_correlations.round(3))

In [ ]:
top_yield_relationships = yield_correlations.head(8)

plt.figure(figsize=(9,5))

sns.barplot(
    x=top_yield_relationships.values,
    y=top_yield_relationships.index
)

plt.title('Strongest Numerical Relationships with Yield')
plt.xlabel('Correlation')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

In [ ]:
environmental_vars = [
    'Rainfall_mm',
    'Avg_Temperature_C',
    'Humidity_pct',
    'Soil_Moisture_pct',
    'Sunlight_Hours_Day',
    'Soil_pH'
]

for col in environmental_vars:
    plt.figure(figsize=(7,5))

    sns.scatterplot(
        data=yield_df,
        x=col,
        y='Yield_Tonnes_Ha',
        alpha=0.5
    )

    sns.regplot(
        data=yield_df,
        x=col,
        y='Yield_Tonnes_Ha',
        scatter=False
    )

    plt.title(f'{col} vs Yield')
    plt.xlabel(col)
    plt.ylabel('Yield (Tonnes/Ha)')
    plt.tight_layout()
    plt.show()

In [ ]:
environmental_correlation = (
    yield_df[
        environmental_vars + ['Yield_Tonnes_Ha']
    ]
    .corr()['Yield_Tonnes_Ha']
    .drop('Yield_Tonnes_Ha')
    .sort_values(key=abs, ascending=False)
)

print("Environmental Variables vs Yield:")
print(environmental_correlation.round(3))

In [ ]:
resource_vars = [
    'Nitrogen_kg_ha',
    'Phosphorus_kg_ha',
    'Potassium_kg_ha',
    'Fertilizer_kg_ha',
    'Pesticide_Litre_ha',
    'Water_Used_m3'
]

resource_correlation = (
    yield_df[
        resource_vars + ['Yield_Tonnes_Ha']
    ]
    .corr()['Yield_Tonnes_Ha']
    .drop('Yield_Tonnes_Ha')
    .sort_values(key=abs, ascending=False)
)

print("Resource Variables vs Yield:")
print(resource_correlation.round(3))

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=yield_df,
    x='Water_Used_m3',
    y='Yield_Tonnes_Ha',
    hue='Season',
    alpha=0.6
)

plt.title('Water Usage vs Agricultural Yield')
plt.xlabel('Water Used (m³)')
plt.ylabel('Yield (Tonnes/Ha)')
plt.tight_layout()
plt.show()

In [ ]:
profit_correlations = (
    yield_df.select_dtypes(include=np.number)
    .corr()['Profit_INR']
    .drop('Profit_INR')
    .sort_values(key=abs, ascending=False)
)

print("Variables most strongly related to Profit:")
print(profit_correlations.round(3))

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=yield_df,
    x='Yield_Tonnes_Ha',
    y='Profit_INR',
    hue='Season',
    alpha=0.6
)

plt.title('Yield vs Profit')
plt.xlabel('Yield (Tonnes/Ha)')
plt.ylabel('Profit (INR)')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import f_oneway

season_groups = [
    group['Yield_Tonnes_Ha'].dropna()
    for _, group in yield_df.groupby('Season')
]

season_names = [
    season for season, _ in yield_df.groupby('Season')
]

anova_result = f_oneway(*season_groups)

print("One-Way ANOVA — Yield Across Seasons")
print("-" * 50)

print("F-statistic:", round(anova_result.statistic, 4))
print("p-value:", anova_result.pvalue)

In [ ]:
alpha = 0.05

if anova_result.pvalue < alpha:
    print("Result: Statistically significant difference.")
    print("Reject the null hypothesis.")
else:
    print("Result: No statistically significant difference.")
    print("Fail to reject the null hypothesis.")

In [ ]:
from scipy.stats import tukey_hsd

tukey_result = tukey_hsd(*season_groups)

print("Tukey HSD — Pairwise Seasonal Yield Comparison")
print("-" * 55)

print(tukey_result)

In [ ]:
print("Season order:")
print(season_names)

print("\nPairwise p-values:")
print(np.round(tukey_result.pvalue, 5))

In [ ]:
profit_groups = [
    group['Profit_INR'].dropna()
    for _, group in yield_df.groupby('Season')
]

profit_anova = f_oneway(*profit_groups)

print("One-Way ANOVA — Profit Across Seasons")
print("-" * 50)

print("F-statistic:", round(profit_anova.statistic, 4))
print("p-value:", profit_anova.pvalue)

if profit_anova.pvalue < 0.05:
    print("\nResult: Profit differs significantly across seasons.")
else:
    print("\nResult: No statistically significant seasonal difference in profit.")

In [ ]:
efficiency_groups = [
    group['Water_Efficiency_t_per_1000m3'].dropna()
    for _, group in clean_df.groupby('Season')
]

efficiency_anova = f_oneway(*efficiency_groups)

print("One-Way ANOVA — Water Efficiency Across Seasons")
print("-" * 55)

print("F-statistic:", round(efficiency_anova.statistic, 4))
print("p-value:", efficiency_anova.pvalue)

if efficiency_anova.pvalue < 0.05:
    print("\nResult: Water efficiency differs significantly across seasons.")
else:
    print("\nResult: No statistically significant seasonal difference.")

In [ ]:
from scipy.stats import pearsonr

print("=" * 60)
print("PEARSON CORRELATION TESTS — ENVIRONMENT vs YIELD")
print("=" * 60)

for col in environmental_vars:

    temp = yield_df[[col, 'Yield_Tonnes_Ha']].dropna()

    r, p = pearsonr(
        temp[col],
        temp['Yield_Tonnes_Ha']
    )

    print(f"\n{col}")
    print(f"Correlation (r): {r:.3f}")
    print(f"p-value: {p:.5f}")

    if p < 0.05:
        print("Significant relationship")
    else:
        print("Not statistically significant")

In [ ]:
print("=" * 60)
print("PEARSON CORRELATION TESTS — RESOURCES vs YIELD")
print("=" * 60)

for col in resource_vars:

    temp = yield_df[[col, 'Yield_Tonnes_Ha']].dropna()

    r, p = pearsonr(
        temp[col],
        temp['Yield_Tonnes_Ha']
    )

    print(f"\n{col}")
    print(f"Correlation (r): {r:.3f}")
    print(f"p-value: {p:.5f}")

    if p < 0.05:
        print("Significant relationship")
    else:
        print("Not statistically significant")

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=yield_df,
    x='Disease_Pest_Risk_pct',
    y='Yield_Tonnes_Ha',
    hue='Season',
    alpha=0.6
)

sns.regplot(
    data=yield_df,
    x='Disease_Pest_Risk_pct',
    y='Yield_Tonnes_Ha',
    scatter=False
)

plt.title('Disease/Pest Risk vs Yield')
plt.xlabel('Disease/Pest Risk (%)')
plt.ylabel('Yield (Tonnes/Ha)')
plt.tight_layout()
plt.show()

In [ ]:
temp = yield_df[
    ['Disease_Pest_Risk_pct', 'Yield_Tonnes_Ha']
].dropna()

r, p = pearsonr(
    temp['Disease_Pest_Risk_pct'],
    temp['Yield_Tonnes_Ha']
)

print("Disease/Pest Risk vs Yield")
print("-" * 40)
print(f"Correlation (r): {r:.3f}")
print(f"p-value: {p:.5f}")

if p < 0.05:
    print("Relationship is statistically significant.")
else:
    print("Relationship is not statistically significant.")

In [ ]:
print("=" * 70)
print("PHASE 5 — STATISTICAL ANALYSIS SUMMARY")
print("=" * 70)

print("\n1. Seasonal Yield ANOVA")
print("F-statistic:", round(anova_result.statistic, 4))
print("p-value:", round(anova_result.pvalue, 6))

print("\n2. Seasonal Profit ANOVA")
print("F-statistic:", round(profit_anova.statistic, 4))
print("p-value:", round(profit_anova.pvalue, 6))

print("\n3. Seasonal Water Efficiency ANOVA")
print("F-statistic:", round(efficiency_anova.statistic, 4))
print("p-value:", round(efficiency_anova.pvalue, 6))

print("\n4. Strongest Environmental Relationships with Yield:")
print(environmental_correlation.head(5).round(3))

print("\n5. Strongest Resource Relationships with Yield:")
print(resource_correlation.head(5).round(3))

Advanced Comparative Analysis

This is the next major phase. We'll focus on the questions in your Problem Statement that involve comparisons across crops, regions, resources, economics and unusual patterns.

In [ ]:
# ============================================================
# PHASE 6.1 — CROP × SEASON ANALYSIS
# ============================================================

print("=" * 70)
print("PHASE 6.1 — CROP × SEASON PERFORMANCE ANALYSIS")
print("=" * 70)

crop_season = (
    yield_df
    .groupby(['Season', 'Crop'])
    .agg(
        Avg_Yield=('Yield_Tonnes_Ha', 'mean'),
        Avg_Profit=('Profit_INR', 'mean'),
        Avg_Revenue=('Revenue_INR', 'mean'),
        Avg_Cost=('Total_Cost_INR', 'mean'),
        Avg_Water_Used=('Water_Used_m3', 'mean'),
        Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean'),
        Avg_Disease_Risk=('Disease_Pest_Risk_pct', 'mean')
    )
    .reset_index()
)

print(crop_season.round(2))

In [ ]:
print("=" * 70)
print("BEST AND WORST PERFORMING CROP BY SEASON")
print("=" * 70)

for season in crop_season['Season'].unique():

    temp = crop_season[crop_season['Season'] == season]

    best = temp.loc[temp['Avg_Yield'].idxmax()]
    worst = temp.loc[temp['Avg_Yield'].idxmin()]

    print(f"\n{season}")
    print(f"Best Crop  : {best['Crop']}")
    print(f"Yield      : {best['Avg_Yield']:.2f} tonnes/ha")

    print(f"Worst Crop : {worst['Crop']}")
    print(f"Yield      : {worst['Avg_Yield']:.2f} tonnes/ha")

In [ ]:
plt.figure(figsize=(10, 6))

yield_heatmap = crop_season.pivot(
    index='Crop',
    columns='Season',
    values='Avg_Yield'
)

sns.heatmap(
    yield_heatmap,
    annot=True,
    fmt='.2f',
    cmap='YlGn'
)

plt.title('Average Crop Yield Across Seasons')
plt.xlabel('Season')
plt.ylabel('Crop')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

profit_heatmap = crop_season.pivot(
    index='Crop',
    columns='Season',
    values='Avg_Profit'
)

sns.heatmap(
    profit_heatmap,
    annot=True,
    fmt='.0f',
    cmap='RdYlGn',
    center=0
)

plt.title('Average Crop Profit Across Seasons')
plt.xlabel('Season')
plt.ylabel('Crop')
plt.tight_layout()
plt.show()

In [ ]:
crop_summary = (
    yield_df
    .groupby('Crop')
    .agg(
        Avg_Yield=('Yield_Tonnes_Ha', 'mean'),
        Avg_Profit=('Profit_INR', 'mean'),
        Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean'),
        Avg_Water_Used=('Water_Used_m3', 'mean')
    )
    .sort_values('Avg_Yield', ascending=False)
)

print("=" * 70)
print("OVERALL CROP PERFORMANCE")
print("=" * 70)

print(crop_summary.round(2))

In [ ]:
print("=" * 70)
print("TOP 5 CROPS BY AVERAGE YIELD")
print("=" * 70)

print(
    crop_summary
    .sort_values('Avg_Yield', ascending=False)
    .head(5)
    .round(2)
)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=crop_summary.reset_index(),
    x='Crop',
    y='Avg_Yield'
)

plt.title('Average Yield by Crop')
plt.xlabel('Crop')
plt.ylabel('Average Yield (Tonnes/Ha)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PHASE 6.2 — STATE × SEASON ANALYSIS
# ============================================================

print("=" * 70)
print("PHASE 6.2 — REGIONAL PERFORMANCE ANALYSIS")
print("=" * 70)

state_season = (
    yield_df
    .groupby(['Season', 'State'])
    .agg(
        Avg_Yield=('Yield_Tonnes_Ha', 'mean'),
        Avg_Profit=('Profit_INR', 'mean'),
        Avg_Revenue=('Revenue_INR', 'mean'),
        Avg_Cost=('Total_Cost_INR', 'mean'),
        Avg_Water_Used=('Water_Used_m3', 'mean'),
        Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean'),
        Avg_Disease_Risk=('Disease_Pest_Risk_pct', 'mean')
    )
    .reset_index()
)

print(state_season.round(2))

In [ ]:
print("=" * 70)
print("BEST AND WORST PERFORMING STATE BY SEASON")
print("=" * 70)

for season in state_season['Season'].unique():

    temp = state_season[state_season['Season'] == season]

    best = temp.loc[temp['Avg_Yield'].idxmax()]
    worst = temp.loc[temp['Avg_Yield'].idxmin()]

    print(f"\n{season}")
    print(f"Best State  : {best['State']}")
    print(f"Yield       : {best['Avg_Yield']:.2f} tonnes/ha")

    print(f"Worst State : {worst['State']}")
    print(f"Yield       : {worst['Avg_Yield']:.2f} tonnes/ha")

In [ ]:
plt.figure(figsize=(12, 8))

state_yield_heatmap = state_season.pivot(
    index='State',
    columns='Season',
    values='Avg_Yield'
)

sns.heatmap(
    state_yield_heatmap,
    annot=True,
    fmt='.2f',
    cmap='YlGn'
)

plt.title('Average Yield by State and Season')
plt.xlabel('Season')
plt.ylabel('State')
plt.tight_layout()
plt.show()

In [ ]:
state_summary = (
    yield_df
    .groupby('State')
    .agg(
        Avg_Yield=('Yield_Tonnes_Ha', 'mean'),
        Avg_Profit=('Profit_INR', 'mean'),
        Avg_Revenue=('Revenue_INR', 'mean'),
        Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean'),
        Avg_Water_Used=('Water_Used_m3', 'mean')
    )
    .sort_values('Avg_Yield', ascending=False)
)

print("=" * 70)
print("OVERALL STATE PERFORMANCE")
print("=" * 70)

print(state_summary.round(2))

In [ ]:
top_states = (
    state_summary
    .sort_values('Avg_Yield', ascending=False)
    .head(10)
)

print("=" * 70)
print("TOP 10 STATES BY AVERAGE YIELD")
print("=" * 70)

print(top_states.round(2))

In [ ]:
print("=" * 70)
print("REGIONAL YIELD CONSISTENCY")
print("=" * 70)

season_state_stats = (
    yield_df
    .groupby('Season')['Yield_Tonnes_Ha']
    .agg(['mean', 'std', 'min', 'max'])
)

season_state_stats['Range'] = (
    season_state_stats['max'] - season_state_stats['min']
)

print(season_state_stats.round(2))

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=top_states.reset_index(),
    x='State',
    y='Avg_Yield'
)

plt.title('Top 10 States by Average Agricultural Yield')
plt.xlabel('State')
plt.ylabel('Average Yield (Tonnes/Ha)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PHASE 6.3 — RESOURCE & WATER EFFICIENCY ANALYSIS
# ============================================================

print("=" * 70)
print("RESOURCE USAGE BY SEASON")
print("=" * 70)

resource_season = (
    clean_df
    .groupby('Season')
    .agg(
        Avg_Water_Used=('Water_Used_m3', 'mean'),
        Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean'),
        Avg_Nitrogen=('Nitrogen_kg_ha', 'mean'),
        Avg_Phosphorus=('Phosphorus_kg_ha', 'mean'),
        Avg_Potassium=('Potassium_kg_ha', 'mean'),
        Avg_Fertilizer=('Fertilizer_kg_ha', 'mean'),
        Avg_Pesticide=('Pesticide_Litre_ha', 'mean')
    )
    .round(2)
)

print(resource_season)

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=resource_season.reset_index(),
    x='Season',
    y='Avg_Water_Used'
)

plt.title('Average Water Usage by Season')
plt.xlabel('Season')
plt.ylabel('Water Used (m³)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=resource_season.reset_index(),
    x='Season',
    y='Avg_Water_Efficiency'
)

plt.title('Water Efficiency by Season')
plt.xlabel('Season')
plt.ylabel('Water Efficiency (Tonnes per 1000 m³)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=yield_df,
    x='Water_Used_m3',
    y='Yield_Tonnes_Ha',
    hue='Season',
    alpha=0.6
)

sns.regplot(
    data=yield_df,
    x='Water_Used_m3',
    y='Yield_Tonnes_Ha',
    scatter=False
)

plt.title('Water Usage vs Agricultural Yield')
plt.xlabel('Water Used (m³)')
plt.ylabel('Yield (Tonnes/Ha)')
plt.tight_layout()
plt.show()

In [ ]:
crop_resources = (
    clean_df
    .groupby('Crop')
    .agg(
        Water_Used=('Water_Used_m3', 'mean'),
        Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean'),
        Nitrogen=('Nitrogen_kg_ha', 'mean'),
        Phosphorus=('Phosphorus_kg_ha', 'mean'),
        Potassium=('Potassium_kg_ha', 'mean'),
        Fertilizer=('Fertilizer_kg_ha', 'mean'),
        Pesticide=('Pesticide_Litre_ha', 'mean')
    )
    .sort_values('Water_Efficiency', ascending=False)
)

print("=" * 70)
print("RESOURCE EFFICIENCY BY CROP")
print("=" * 70)

print(crop_resources.round(2))

In [ ]:
print("=" * 70)
print("TOP 5 CROPS BY WATER EFFICIENCY")
print("=" * 70)

print(
    crop_resources
    .sort_values('Water_Efficiency', ascending=False)
    .head(5)
    .round(2)
)

In [ ]:
irrigation_analysis = (
    clean_df
    .groupby('Irrigation_Method')
    .agg(
        Avg_Yield=('Yield_Tonnes_Ha', 'mean'),
        Avg_Water_Used=('Water_Used_m3', 'mean'),
        Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean'),
        Avg_Profit=('Profit_INR', 'mean')
    )
    .sort_values('Avg_Water_Efficiency', ascending=False)
)

print("=" * 70)
print("IRRIGATION METHOD PERFORMANCE")
print("=" * 70)

print(irrigation_analysis.round(2))

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=irrigation_analysis.reset_index(),
    x='Irrigation_Method',
    y='Avg_Water_Efficiency'
)

plt.title('Water Efficiency by Irrigation Method')
plt.xlabel('Irrigation Method')
plt.ylabel('Water Efficiency')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 70)
print("RESOURCE & WATER EFFICIENCY SUMMARY")
print("=" * 70)

best_water_season = resource_season['Avg_Water_Efficiency'].idxmax()
highest_water_season = resource_season['Avg_Water_Used'].idxmax()

print(f"Most Water-Efficient Season : {best_water_season}")
print(f"Highest Water Usage Season  : {highest_water_season}")

best_crop = crop_resources['Water_Efficiency'].idxmax()
print(f"Most Water-Efficient Crop   : {best_crop}")

best_irrigation = irrigation_analysis['Avg_Water_Efficiency'].idxmax()
print(f"Most Efficient Irrigation   : {best_irrigation}")

In [ ]:
# ============================================================
# PHASE 6.4 — ECONOMIC PERFORMANCE ANALYSIS
# ============================================================

print("=" * 70)
print("ECONOMIC PERFORMANCE BY SEASON")
print("=" * 70)

economic_season = (
    clean_df
    .groupby('Season')
    .agg(
        Avg_Revenue=('Revenue_INR', 'mean'),
        Avg_Cost=('Total_Cost_INR', 'mean'),
        Avg_Profit=('Profit_INR', 'mean')
    )
)

economic_season['Profit_Margin_pct'] = (
    economic_season['Avg_Profit'] /
    economic_season['Avg_Revenue'] * 100
)

print(economic_season.round(2))

In [ ]:
economic_plot = economic_season.reset_index().melt(
    id_vars='Season',
    value_vars=['Avg_Revenue', 'Avg_Cost', 'Avg_Profit'],
    var_name='Metric',
    value_name='Amount'
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=economic_plot,
    x='Season',
    y='Amount',
    hue='Metric'
)

plt.title('Economic Performance Across Seasons')
plt.xlabel('Season')
plt.ylabel('Amount (INR)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=economic_season.reset_index(),
    x='Season',
    y='Profit_Margin_pct'
)

plt.title('Profit Margin by Season')
plt.xlabel('Season')
plt.ylabel('Profit Margin (%)')
plt.tight_layout()
plt.show()

In [ ]:
crop_economic = (
    clean_df
    .groupby('Crop')
    .agg(
        Avg_Revenue=('Revenue_INR', 'mean'),
        Avg_Cost=('Total_Cost_INR', 'mean'),
        Avg_Profit=('Profit_INR', 'mean')
    )
)

crop_economic['Profit_Margin_pct'] = (
    crop_economic['Avg_Profit'] /
    crop_economic['Avg_Revenue'] * 100
)

crop_economic = crop_economic.sort_values(
    'Avg_Profit',
    ascending=False
)

print("=" * 70)
print("ECONOMIC PERFORMANCE BY CROP")
print("=" * 70)

print(crop_economic.round(2))

In [ ]:
print("=" * 70)
print("TOP 5 CROPS BY AVERAGE PROFIT")
print("=" * 70)

print(
    crop_economic
    .sort_values('Avg_Profit', ascending=False)
    .head(5)
    .round(2)
)

In [ ]:
crop_season_economic = (
    clean_df
    .groupby(['Season', 'Crop'])
    .agg(
        Avg_Revenue=('Revenue_INR', 'mean'),
        Avg_Cost=('Total_Cost_INR', 'mean'),
        Avg_Profit=('Profit_INR', 'mean')
    )
    .reset_index()
)

crop_season_economic['Profit_Margin_pct'] = (
    crop_season_economic['Avg_Profit'] /
    crop_season_economic['Avg_Revenue'] * 100
)

print("=" * 70)
print("CROP × SEASON ECONOMIC PERFORMANCE")
print("=" * 70)

print(crop_season_economic.round(2))

In [ ]:
print("=" * 70)
print("MOST PROFITABLE CROP BY SEASON")
print("=" * 70)

for season in crop_season_economic['Season'].unique():

    temp = crop_season_economic[
        crop_season_economic['Season'] == season
    ]

    best = temp.loc[temp['Avg_Profit'].idxmax()]
    worst = temp.loc[temp['Avg_Profit'].idxmin()]

    print(f"\n{season}")
    print(f"Most Profitable : {best['Crop']}")
    print(f"Average Profit  : ₹{best['Avg_Profit']:,.2f}")

    print(f"Least Profitable: {worst['Crop']}")
    print(f"Average Profit  : ₹{worst['Avg_Profit']:,.2f}")

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=clean_df,
    x='Total_Cost_INR',
    y='Revenue_INR',
    hue='Season',
    size='Profit_INR',
    alpha=0.6
)

plt.title('Revenue vs Cost with Profit Indication')
plt.xlabel('Total Cost (INR)')
plt.ylabel('Revenue (INR)')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 70)
print("ECONOMIC PERFORMANCE SUMMARY")
print("=" * 70)

best_profit_season = economic_season['Avg_Profit'].idxmax()
best_margin_season = economic_season['Profit_Margin_pct'].idxmax()
highest_revenue_season = economic_season['Avg_Revenue'].idxmax()

print(f"Highest Average Revenue : {highest_revenue_season}")
print(f"Highest Average Profit  : {best_profit_season}")
print(f"Highest Profit Margin   : {best_margin_season}")

print("\nProfit Margin by Season:")
print(
    economic_season['Profit_Margin_pct']
    .sort_values(ascending=False)
    .round(2)
)

In [ ]:
# ============================================================
# PHASE 6.5 — UNUSUAL PATTERNS & RISK ANALYSIS
# ============================================================

print("=" * 70)
print("HIGH WATER USAGE + LOW YIELD")
print("=" * 70)

water_median = yield_df['Water_Used_m3'].median()
yield_median = yield_df['Yield_Tonnes_Ha'].median()

high_water_low_yield = yield_df[
    (yield_df['Water_Used_m3'] > water_median) &
    (yield_df['Yield_Tonnes_Ha'] < yield_median)
].copy()

print("Number of observations:", len(high_water_low_yield))
print("\nPercentage of observations:",
      round(len(high_water_low_yield) / len(yield_df) * 100, 2), "%")

print("\nBy season:")
print(
    high_water_low_yield['Season']
    .value_counts()
)

In [ ]:
print("=" * 70)
print("HIGH COST + LOW PROFIT")
print("=" * 70)

cost_median = clean_df['Total_Cost_INR'].median()
profit_median = clean_df['Profit_INR'].median()

high_cost_low_profit = clean_df[
    (clean_df['Total_Cost_INR'] > cost_median) &
    (clean_df['Profit_INR'] < profit_median)
].copy()

print("Number of observations:", len(high_cost_low_profit))

print("\nPercentage of observations:",
      round(len(high_cost_low_profit) / len(clean_df) * 100, 2), "%")

print("\nBy season:")
print(
    high_cost_low_profit['Season']
    .value_counts()
)

In [ ]:
print("=" * 70)
print("HIGH DISEASE/PEST RISK + LOW YIELD")
print("=" * 70)

risk_median = yield_df['Disease_Pest_Risk_pct'].median()

high_risk_low_yield = yield_df[
    (yield_df['Disease_Pest_Risk_pct'] > risk_median) &
    (yield_df['Yield_Tonnes_Ha'] < yield_median)
].copy()

print("Number of observations:", len(high_risk_low_yield))

print("\nPercentage of observations:",
      round(len(high_risk_low_yield) / len(yield_df) * 100, 2), "%")

print("\nBy season:")
print(
    high_risk_low_yield['Season']
    .value_counts()
)

In [ ]:
print("=" * 70)
print("NEGATIVE PROFIT ANALYSIS")
print("=" * 70)

negative_profit = clean_df[
    clean_df['Profit_INR'] < 0
].copy()

print("Negative-profit observations:", len(negative_profit))

print("Percentage:",
      round(len(negative_profit) / len(clean_df) * 100, 2), "%")

print("\nNegative profit by season:")
print(
    negative_profit['Season']
    .value_counts()
)

print("\nAverage profit of negative-profit observations:")
print(
    negative_profit
    .groupby('Season')['Profit_INR']
    .mean()
    .round(2)
)

In [ ]:
print("=" * 70)
print("BEST AND WORST CROP × SEASON COMBINATIONS")
print("=" * 70)

combination_summary = (
    yield_df
    .groupby(['Season', 'Crop'])
    .agg(
        Avg_Yield=('Yield_Tonnes_Ha', 'mean'),
        Avg_Profit=('Profit_INR', 'mean'),
        Avg_Water_Efficiency=(
            'Water_Efficiency_t_per_1000m3',
            'mean'
        ),
        Avg_Disease_Risk=('Disease_Pest_Risk_pct', 'mean')
    )
    .reset_index()
)

print("\nTOP 10 BY YIELD")
print(
    combination_summary
    .sort_values('Avg_Yield', ascending=False)
    .head(10)
    .round(2)
)

print("\nBOTTOM 10 BY YIELD")
print(
    combination_summary
    .sort_values('Avg_Yield', ascending=True)
    .head(10)
    .round(2)
)

In [ ]:
print("=" * 70)
print("LOW-PROFIT CROP × SEASON COMBINATIONS")
print("=" * 70)

print(
    combination_summary
    .sort_values('Avg_Profit', ascending=True)
    .head(10)
    .round(2)
)

In [ ]:
print("=" * 70)
print("UNUSUAL PATTERNS & RISK SUMMARY")
print("=" * 70)

print(f"High water + low yield cases : {len(high_water_low_yield)}")
print(f"High cost + low profit cases : {len(high_cost_low_profit)}")
print(f"High risk + low yield cases  : {len(high_risk_low_yield)}")
print(f"Negative profit cases        : {len(negative_profit)}")

print("\nMost common season among negative-profit cases:")

if len(negative_profit) > 0:
    print(negative_profit['Season'].value_counts().idxmax())
else:
    print("No negative-profit observations.")

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=clean_df,
    x='Total_Cost_INR',
    y='Profit_INR',
    hue='Season',
    alpha=0.6
)

plt.axhline(0, linestyle='--')

plt.title('Cost vs Profit Across Seasons')
plt.xlabel('Total Cost (INR)')
plt.ylabel('Profit (INR)')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PHASE 6.6 — OVERALL PERFORMANCE RANKING
# ============================================================

print("=" * 70)
print("PHASE 6.6 — OVERALL PERFORMANCE RANKING")
print("=" * 70)

ranking_df = (
    yield_df
    .groupby(['Season', 'Crop'])
    .agg(
        Avg_Yield=('Yield_Tonnes_Ha', 'mean'),
        Avg_Profit=('Profit_INR', 'mean'),
        Avg_Water_Efficiency=(
            'Water_Efficiency_t_per_1000m3',
            'mean'
        )
    )
    .reset_index()
)

# Min-Max normalization
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

ranking_df['Yield_Score'] = minmax(
    ranking_df['Avg_Yield']
)

ranking_df['Profit_Score'] = minmax(
    ranking_df['Avg_Profit']
)

ranking_df['Water_Efficiency_Score'] = minmax(
    ranking_df['Avg_Water_Efficiency']
)

print(ranking_df.round(3))

In [ ]:
ranking_df['Overall_Score'] = (
    ranking_df['Yield_Score'] * 0.33 +
    ranking_df['Profit_Score'] * 0.33 +
    ranking_df['Water_Efficiency_Score'] * 0.34
)

ranking_df = ranking_df.sort_values(
    'Overall_Score',
    ascending=False
).reset_index(drop=True)

ranking_df['Rank'] = range(1, len(ranking_df) + 1)

print("=" * 70)
print("OVERALL CROP × SEASON RANKING")
print("=" * 70)

print(
    ranking_df[
        [
            'Rank',
            'Season',
            'Crop',
            'Avg_Yield',
            'Avg_Profit',
            'Avg_Water_Efficiency',
            'Overall_Score'
        ]
    ].round(3)
)

In [ ]:
print("=" * 70)
print("TOP 10 CROP × SEASON COMBINATIONS")
print("=" * 70)

top_10 = ranking_df.head(10)

print(
    top_10[
        [
            'Rank',
            'Season',
            'Crop',
            'Avg_Yield',
            'Avg_Profit',
            'Avg_Water_Efficiency',
            'Overall_Score'
        ]
    ].round(3)
)

In [ ]:
print("=" * 70)
print("BOTTOM 10 CROP × SEASON COMBINATIONS")
print("=" * 70)

bottom_10 = ranking_df.tail(10)

print(
    bottom_10[
        [
            'Rank',
            'Season',
            'Crop',
            'Avg_Yield',
            'Avg_Profit',
            'Avg_Water_Efficiency',
            'Overall_Score'
        ]
    ].round(3)
)

In [ ]:
season_ranking = (
    ranking_df
    .groupby('Season')
    .agg(
        Avg_Yield=('Avg_Yield', 'mean'),
        Avg_Profit=('Avg_Profit', 'mean'),
        Avg_Water_Efficiency=('Avg_Water_Efficiency', 'mean'),
        Overall_Score=('Overall_Score', 'mean')
    )
    .sort_values('Overall_Score', ascending=False)
)

print("=" * 70)
print("OVERALL SEASON RANKING")
print("=" * 70)

print(season_ranking.round(3))

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=ranking_df.head(10),
    x='Overall_Score',
    y='Crop',
    hue='Season'
)

plt.title('Top 10 Crop × Season Combinations by Overall Performance')
plt.xlabel('Overall Performance Score')
plt.ylabel('Crop')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 70)
print("FINAL PERFORMANCE RANKING SUMMARY")
print("=" * 70)

best = ranking_df.iloc[0]
worst = ranking_df.iloc[-1]

print(f"Best Combination  : {best['Crop']} ({best['Season']})")
print(f"Overall Score     : {best['Overall_Score']:.3f}")

print(f"\nLowest Combination: {worst['Crop']} ({worst['Season']})")
print(f"Overall Score     : {worst['Overall_Score']:.3f}")

print("\nBest Overall Season:")
print(season_ranking.index[0])

🚀Final KPIs & Professional Visualizations

Now we convert the analysis into a clean final-results section. Run these cells in order.

In [ ]:
# ============================================================
# PHASE 7 — FINAL AGRICULTURAL PERFORMANCE KPIs
# ============================================================

print("=" * 75)
print("PHASE 7 — FINAL AGRICULTURAL PERFORMANCE KPIs")
print("=" * 75)

# Create a fresh season summary
final_season_kpi = (
    yield_df
    .groupby('Season')
    .agg(
        Avg_Yield=('Yield_Tonnes_Ha', 'mean'),
        Avg_Profit=('Profit_INR', 'mean'),
        Avg_Water_Efficiency=(
            'Water_Efficiency_t_per_1000m3',
            'mean'
        )
    )
)

# Identify best seasons
best_yield_season = final_season_kpi['Avg_Yield'].idxmax()
best_profit_season = final_season_kpi['Avg_Profit'].idxmax()
best_efficiency_season = final_season_kpi['Avg_Water_Efficiency'].idxmax()

# Best and lowest crop-season combinations
final_ranking = (
    yield_df
    .groupby(['Season', 'Crop'])
    .agg(
        Avg_Yield=('Yield_Tonnes_Ha', 'mean'),
        Avg_Profit=('Profit_INR', 'mean'),
        Avg_Water_Efficiency=(
            'Water_Efficiency_t_per_1000m3',
            'mean'
        )
    )
    .reset_index()
)

# Normalize
for col in ['Avg_Yield', 'Avg_Profit', 'Avg_Water_Efficiency']:
    min_val = final_ranking[col].min()
    max_val = final_ranking[col].max()

    if max_val != min_val:
        final_ranking[col + '_Score'] = (
            (final_ranking[col] - min_val) /
            (max_val - min_val)
        )
    else:
        final_ranking[col + '_Score'] = 0

# Overall score
final_ranking['Overall_Score'] = (
    final_ranking['Avg_Yield_Score'] * 0.33 +
    final_ranking['Avg_Profit_Score'] * 0.33 +
    final_ranking['Avg_Water_Efficiency_Score'] * 0.34
)

final_ranking = final_ranking.sort_values(
    'Overall_Score',
    ascending=False
).reset_index(drop=True)

best = final_ranking.iloc[0]
lowest = final_ranking.iloc[-1]

print(f"\nBest Yield Season         : {best_yield_season}")
print(f"Average Yield             : {final_season_kpi.loc[best_yield_season, 'Avg_Yield']:.2f} tonnes/ha")

print(f"\nHighest Profit Season     : {best_profit_season}")
print(f"Average Profit            : ₹{final_season_kpi.loc[best_profit_season, 'Avg_Profit']:,.2f}")

print(f"\nBest Water Efficiency     : {best_efficiency_season}")
print(f"Water Efficiency          : {final_season_kpi.loc[best_efficiency_season, 'Avg_Water_Efficiency']:.2f}")

print(f"\nBest Crop × Season        : {best['Crop']} ({best['Season']})")
print(f"Overall Score             : {best['Overall_Score']:.3f}")

print(f"\nLowest Crop × Season      : {lowest['Crop']} ({lowest['Season']})")
print(f"Overall Score             : {lowest['Overall_Score']:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use the fresh KPI table created in Cell 1
plot_df = final_season_kpi.copy()

# ------------------------------------------------------------
# 1. Average Yield by Season
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
sns.barplot(
    data=plot_df,
    x='Season',
    y='Avg_Yield'
)
plt.title('Average Yield by Season')
plt.xlabel('Season')
plt.ylabel('Average Yield (Tonnes/Ha)')
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 2. Average Profit by Season
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
sns.barplot(
    data=plot_df,
    x='Season',
    y='Avg_Profit'
)
plt.title('Average Profit by Season')
plt.xlabel('Season')
plt.ylabel('Average Profit (INR)')
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 3. Water Efficiency by Season
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
sns.barplot(
    data=plot_df,
    x='Season',
    y='Avg_Water_Efficiency'
)
plt.title('Water Efficiency by Season')
plt.xlabel('Season')
plt.ylabel('Water Efficiency (Tonnes / 1000 m³)')
plt.tight_layout()
plt.show()


print("COMPLETED SUCCESSFULLY")

In [ ]:
# ============================================================
# YIELD DISTRIBUTION BY SEASON
# ============================================================

plt.figure(figsize=(9, 6))

sns.boxplot(
    data=yield_df,
    x='Season',
    y='Yield_Tonnes_Ha'
)

plt.title('Yield Distribution Across Seasons')
plt.xlabel('Season')
plt.ylabel('Yield (Tonnes/Ha)')
plt.tight_layout()
plt.show()

print("COMPLETED")

In [ ]:
# ============================================================
#CROP × SEASON YIELD HEATMAP
# ============================================================

yield_heatmap = yield_df.pivot_table(
    index='Crop',
    columns='Season',
    values='Yield_Tonnes_Ha',
    aggfunc='mean'
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    yield_heatmap,
    annot=True,
    fmt='.2f',
    cmap='YlGnBu'
)

plt.title('Average Crop Yield Across Seasons')
plt.xlabel('Season')
plt.ylabel('Crop')
plt.tight_layout()
plt.show()

print("COMPLETED")

In [ ]:
# ============================================================
# CROP × SEASON PROFIT HEATMAP
# ============================================================

profit_heatmap = clean_df.pivot_table(
    index='Crop',
    columns='Season',
    values='Profit_INR',
    aggfunc='mean'
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    profit_heatmap,
    annot=True,
    fmt='.0f',
    cmap='RdYlGn',
    center=0
)

plt.title('Average Crop Profit Across Seasons')
plt.xlabel('Season')
plt.ylabel('Crop')
plt.tight_layout()
plt.show()

print("COMPLETED")

In [ ]:
# ============================================================
# WATER EFFICIENCY BY CROP
# ============================================================

water_crop = clean_df.groupby('Crop').agg(
    Avg_Water_Used=('Water_Used_m3', 'mean'),
    Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean')
).sort_values(
    'Avg_Water_Efficiency',
    ascending=False
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=water_crop.reset_index(),
    x='Avg_Water_Efficiency',
    y='Crop'
)

plt.title('Average Water Efficiency by Crop')
plt.xlabel('Water Efficiency (Tonnes / 1000 m³)')
plt.ylabel('Crop')
plt.tight_layout()
plt.show()

print("Top 5 Water-Efficient Crops:")
display(water_crop.head(5))

print("COMPLETED")

In [ ]:
# ============================================================
# WATER USAGE VS YIELD
# ============================================================

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=yield_df,
    x='Water_Used_m3',
    y='Yield_Tonnes_Ha',
    hue='Season',
    alpha=0.6
)

plt.title('Water Usage vs Agricultural Yield')
plt.xlabel('Water Used (m³)')
plt.ylabel('Yield (Tonnes/Ha)')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# RESOURCE CORRELATION WITH YIELD
# ============================================================

resource_cols = [
    'Nitrogen_kg_ha',
    'Phosphorus_kg_ha',
    'Potassium_kg_ha',
    'Fertilizer_kg_ha',
    'Pesticide_Litre_ha',
    'Water_Used_m3'
]

resource_corr = yield_df[resource_cols + ['Yield_Tonnes_Ha']].corr()

yield_resource_corr = (
    resource_corr['Yield_Tonnes_Ha']
    .drop('Yield_Tonnes_Ha')
    .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))

sns.barplot(
    x=yield_resource_corr.values,
    y=yield_resource_corr.index
)

plt.title('Resource Correlation with Yield')
plt.xlabel('Pearson Correlation')
plt.ylabel('Resource')
plt.axvline(0, linewidth=1)
plt.tight_layout()
plt.show()

print("Resource-Yield Correlations:")
display(yield_resource_corr)


In [ ]:
# ============================================================
# ENVIRONMENTAL CORRELATION WITH YIELD
# ============================================================

environment_cols = [
    'Rainfall_mm',
    'Avg_Temperature_C',
    'Humidity_pct',
    'Sunlight_Hours_Day',
    'Soil_pH',
    'Soil_Moisture_pct'
]

environment_corr = yield_df[
    environment_cols + ['Yield_Tonnes_Ha']
].corr()

yield_environment_corr = (
    environment_corr['Yield_Tonnes_Ha']
    .drop('Yield_Tonnes_Ha')
    .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))

sns.barplot(
    x=yield_environment_corr.values,
    y=yield_environment_corr.index
)

plt.title('Environmental Correlation with Yield')
plt.xlabel('Pearson Correlation')
plt.ylabel('Environmental Factor')
plt.axvline(0, linewidth=1)
plt.tight_layout()
plt.show()

print("Environmental-Yield Correlations:")
display(yield_environment_corr)


In [ ]:
# ============================================================
# REVENUE VS PROFIT
# ============================================================

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=clean_df,
    x='Revenue_INR',
    y='Profit_INR',
    hue='Season',
    alpha=0.6
)

plt.axhline(0, linewidth=1)

plt.title('Revenue vs Profit Across Seasons')
plt.xlabel('Revenue (INR)')
plt.ylabel('Profit (INR)')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
#  FINAL KPI TABLE
# ============================================================

final_kpi_table = final_season_kpi.reset_index().copy()

final_kpi_table['Yield_Rank'] = (
    final_kpi_table['Avg_Yield']
    .rank(ascending=False)
    .astype(int)
)

final_kpi_table['Profit_Rank'] = (
    final_kpi_table['Avg_Profit']
    .rank(ascending=False)
    .astype(int)
)

final_kpi_table['Water_Efficiency_Rank'] = (
    final_kpi_table['Avg_Water_Efficiency']
    .rank(ascending=False)
    .astype(int)
)

print("FINAL SEASON KPI TABLE")

display(final_kpi_table)

print("\nBest Season by Yield:",
      final_kpi_table.loc[
          final_kpi_table['Avg_Yield'].idxmax(),
          'Season'
      ])

print("Best Season by Profit:",
      final_kpi_table.loc[
          final_kpi_table['Avg_Profit'].idxmax(),
          'Season'
      ])

print("Best Season by Water Efficiency:",
      final_kpi_table.loc[
          final_kpi_table['Avg_Water_Efficiency'].idxmax(),
          'Season'
      ])


In [ ]:
# ============================================================
# FINAL PROJECT INSIGHTS
# ============================================================

# Convert Season index into a normal column
kpi_df = final_season_kpi.reset_index().copy()

best_yield_season = kpi_df.loc[
    kpi_df['Avg_Yield'].idxmax(), 'Season'
]

best_profit_season = kpi_df.loc[
    kpi_df['Avg_Profit'].idxmax(), 'Season'
]

best_efficiency_season = kpi_df.loc[
    kpi_df['Avg_Water_Efficiency'].idxmax(), 'Season'
]

# Best and worst crop-season combination
best_combination = final_ranking.loc[
    final_ranking['Overall_Score'].idxmax()
]

worst_combination = final_ranking.loc[
    final_ranking['Overall_Score'].idxmin()
]

print("=" * 60)
print("FINAL AGRICULTURAL PERFORMANCE INSIGHTS")
print("=" * 60)

print(f"\n1. Best season by average yield      : {best_yield_season}")
print(f"2. Best season by average profit     : {best_profit_season}")
print(f"3. Best season by water efficiency   : {best_efficiency_season}")

print(
    f"\n4. Best crop-season combination      : "
    f"{best_combination['Crop']} ({best_combination['Season']})"
)

print(
    f"5. Lowest crop-season combination    : "
    f"{worst_combination['Crop']} ({worst_combination['Season']})"
)

print(
    f"\n6. Best combination overall score    : "
    f"{best_combination['Overall_Score']:.3f}"
)

print(
    f"7. Lowest combination overall score  : "
    f"{worst_combination['Overall_Score']:.3f}"
)

print("\n" + "=" * 60)
print("IMPORTANT STATISTICAL FINDINGS")
print("=" * 60)

print("\n• Seasonal yield differences were NOT statistically significant.")
print("• Seasonal profit differences WERE statistically significant.")
print("• Seasonal water-efficiency differences WERE statistically significant.")
print("• Water usage showed the strongest resource-level association with yield.")
print("• Environmental variables showed weak linear relationships with yield.")
print("• Correlation does not imply causation.")

print("\nPHASE 7 COMPLETED SUCCESSFULLY")

# Key Findings, Recommendations and Conclusion

This section summarizes the major findings obtained from the seasonal
agriculture performance analysis.

The findings combine exploratory analysis, seasonal comparisons,
resource and environmental analysis, economic analysis, correlation
analysis, and statistical testing.

The objective is to convert the analytical results into clear,
evidence-based agricultural insights and recommendations.

## 1 Key Findings

The analysis identified several important patterns in agricultural
performance across Kharif, Rabi and Zaid seasons.

The major performance indicators considered were:

- Average yield
- Average profit
- Water efficiency
- Resource usage
- Environmental conditions
- Crop-season performance
- Economic performance
- Disease and pest risk

The following results summarize the most important observations.

In [ ]:
# ============================================================
# KEY FINDINGS — SUMMARY
# ============================================================

kpi_df = final_season_kpi.reset_index().copy()

best_yield = kpi_df.loc[kpi_df['Avg_Yield'].idxmax()]
best_profit = kpi_df.loc[kpi_df['Avg_Profit'].idxmax()]
best_efficiency = kpi_df.loc[
    kpi_df['Avg_Water_Efficiency'].idxmax()
]

best_combination = final_ranking.loc[
    final_ranking['Overall_Score'].idxmax()
]

worst_combination = final_ranking.loc[
    final_ranking['Overall_Score'].idxmin()
]

print("KEY FINDINGS")
print("=" * 60)

print(f"Best Yield Season       : {best_yield['Season']}")
print(f"Highest Average Yield   : {best_yield['Avg_Yield']:.3f} tonnes/ha")

print(f"\nHighest Profit Season   : {best_profit['Season']}")
print(f"Highest Average Profit  : ₹{best_profit['Avg_Profit']:,.2f}")

print(f"\nBest Water Efficiency   : {best_efficiency['Season']}")
print(
    f"Water Efficiency        : "
    f"{best_efficiency['Avg_Water_Efficiency']:.3f}"
)

print(
    f"\nBest Crop-Season        : "
    f"{best_combination['Crop']} ({best_combination['Season']})"
)

print(
    f"Best Overall Score      : "
    f"{best_combination['Overall_Score']:.3f}"
)

print(
    f"\nLowest Crop-Season      : "
    f"{worst_combination['Crop']} ({worst_combination['Season']})"
)

print(
    f"Lowest Overall Score    : "
    f"{worst_combination['Overall_Score']:.3f}"
)

### Interpretation

Kharif demonstrated the strongest descriptive performance across the
three major indicators: yield, profit and water efficiency.

The crop-season ranking identified Sugarcane during Kharif as the
highest-performing combination according to the project's composite
performance score.

Rice during Zaid recorded the lowest composite score.

These rankings represent patterns observed in the dataset and should
not be interpreted as causal relationships.

## 2 Seasonal Performance Comparison

Seasonal performance is compared using three important indicators:

- Average Yield
- Average Profit
- Average Water Efficiency

Using separate visualizations makes it easier to identify differences
between agricultural seasons.

In [ ]:
# ============================================================
# SEASONAL PERFORMANCE VISUALIZATION
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(
    data=kpi_df,
    x='Season',
    y='Avg_Yield',
    ax=axes[0]
)

axes[0].set_title('Average Yield by Season')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Yield (Tonnes/Ha)')


sns.barplot(
    data=kpi_df,
    x='Season',
    y='Avg_Profit',
    ax=axes[1]
)

axes[1].set_title('Average Profit by Season')
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Profit (INR)')


sns.barplot(
    data=kpi_df,
    x='Season',
    y='Avg_Water_Efficiency',
    ax=axes[2]
)

axes[2].set_title('Water Efficiency by Season')
axes[2].set_xlabel('Season')
axes[2].set_ylabel('Tonnes / 1000 m³')

plt.suptitle(
    'Seasonal Agricultural Performance Dashboard',
    fontsize=16,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

### Interpretation

Kharif shows the highest values across yield, profit and water
efficiency.

Rabi demonstrates intermediate performance, while Zaid records the
lowest values across the three indicators.

The negative average profit observed for Zaid highlights an important
economic concern that requires further investigation.

## 3 Statistical Findings

Statistical hypothesis testing was performed to determine whether
observed seasonal differences were statistically meaningful.

Analysis of Variance (ANOVA) was used for:

- Seasonal yield
- Seasonal profit
- Seasonal water efficiency

A significance level of 0.05 was considered.

In [ ]:
# ============================================================
# STATISTICAL FINDINGS SUMMARY
# ============================================================

from scipy.stats import f_oneway

yield_groups = [
    group['Yield_Tonnes_Ha'].dropna()
    for _, group in yield_df.groupby('Season')
]

profit_groups = [
    group['Profit_INR'].dropna()
    for _, group in clean_df.groupby('Season')
]

efficiency_groups = [
    group['Water_Efficiency_t_per_1000m3'].dropna()
    for _, group in clean_df.groupby('Season')
]

yield_anova = f_oneway(*yield_groups)
profit_anova = f_oneway(*profit_groups)
efficiency_anova = f_oneway(*efficiency_groups)

anova_results = pd.DataFrame({
    'Metric': [
        'Yield',
        'Profit',
        'Water Efficiency'
    ],
    'F_Statistic': [
        yield_anova.statistic,
        profit_anova.statistic,
        efficiency_anova.statistic
    ],
    'P_Value': [
        yield_anova.pvalue,
        profit_anova.pvalue,
        efficiency_anova.pvalue
    ]
})

anova_results['Significant'] = (
    anova_results['P_Value'] < 0.05
)

display(anova_results)

### Statistical Interpretation

The ANOVA results indicate that:

- Seasonal yield differences were not statistically significant.
- Seasonal profit differences were statistically significant.
- Seasonal water-efficiency differences were statistically significant.

Therefore, the higher observed average yield during Kharif should be
treated as a descriptive pattern rather than statistically confirmed
seasonal superiority in yield.

In contrast, the differences observed in profit and water efficiency
provide stronger statistical evidence of seasonal variation.

## 4 Resource and Environmental Insights

The relationship between agricultural yield and resource/environmental
variables was examined using Pearson correlation.

This analysis helps identify variables that are associated with yield.

Correlation measures association and does not establish causation.

In [ ]:
# ============================================================
# RESOURCE + ENVIRONMENT CORRELATION VISUALIZATION
# ============================================================

resource_cols = [
    'Nitrogen_kg_ha',
    'Phosphorus_kg_ha',
    'Potassium_kg_ha',
    'Fertilizer_kg_ha',
    'Pesticide_Litre_ha',
    'Water_Used_m3'
]

environment_cols = [
    'Rainfall_mm',
    'Avg_Temperature_C',
    'Humidity_pct',
    'Sunlight_Hours_Day',
    'Soil_pH',
    'Soil_Moisture_pct'
]

resource_corr = (
    yield_df[resource_cols + ['Yield_Tonnes_Ha']]
    .corr()['Yield_Tonnes_Ha']
    .drop('Yield_Tonnes_Ha')
    .sort_values()
)

environment_corr = (
    yield_df[environment_cols + ['Yield_Tonnes_Ha']]
    .corr()['Yield_Tonnes_Ha']
    .drop('Yield_Tonnes_Ha')
    .sort_values()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    x=resource_corr.values,
    y=resource_corr.index,
    ax=axes[0]
)

axes[0].set_title('Resource Variables vs Yield')
axes[0].set_xlabel('Pearson Correlation')
axes[0].set_ylabel('Resource')


sns.barplot(
    x=environment_corr.values,
    y=environment_corr.index,
    ax=axes[1]
)

axes[1].set_title('Environmental Variables vs Yield')
axes[1].set_xlabel('Pearson Correlation')
axes[1].set_ylabel('Environmental Factor')

plt.suptitle(
    'Factors Associated with Agricultural Yield',
    fontsize=16,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

### Interpretation

Water usage showed the strongest observed resource-level association
with yield.

Nitrogen and phosphorus showed statistically significant but weak
positive relationships with yield.

The environmental variables examined individually showed weak linear
relationships with yield.

These results suggest that agricultural yield is likely influenced by
multiple interacting factors rather than a single environmental or
resource variable.

## 5 Evidence-Based Recommendations

Based on the analytical and statistical findings, the following
recommendations are proposed:

1. Use seasonal performance data to support agricultural planning.

2. Closely monitor water consumption, particularly during seasons with
   lower water efficiency.

3. Investigate the causes of negative average profit during Zaid.

4. Evaluate crop-season combinations individually rather than relying
   only on overall seasonal averages.

5. Include water efficiency as an important agricultural performance
   indicator.

6. Review high-cost and low-profit farming cases to identify potential
   efficiency improvements.

7. Consider crop-specific resource management instead of assuming that
   increased input automatically produces higher yield.

8. Use statistical testing before treating observed differences as
   significant.

9. Collect additional environmental, soil and farm-management data for
   future predictive analysis.

10. Future work can apply machine-learning models to predict yield,
    profit and agricultural risk.

## 6 Conclusion

The Seasonal Agriculture Performance Analysis successfully examined
agricultural performance across seasons, crops and regions using
production, resource, environmental and economic indicators.

Kharif recorded the highest observed average yield, profit and water
efficiency, while Zaid recorded the weakest overall descriptive
performance and a negative average profit.

Statistical testing showed that seasonal differences in yield were not
statistically significant. However, seasonal differences in profit and
water efficiency were statistically significant.

The crop-season analysis identified Sugarcane during Kharif as the
highest-ranked combination according to the project's composite
performance score.

Overall, the project demonstrates how data cleaning, exploratory data
analysis, visualization, correlation analysis and statistical testing
can be combined to generate evidence-based agricultural insights.

The results can support seasonal planning, resource-efficiency
assessment and identification of crop-season combinations requiring
further investigation.

In [ ]:
# ============================================================
# FINAL PROJECT KPI DASHBOARD
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Yield
sns.barplot(
    data=kpi_df,
    x='Season',
    y='Avg_Yield',
    ax=axes[0]
)
axes[0].set_title('Average Yield')
axes[0].set_ylabel('Tonnes/Ha')


# Profit
sns.barplot(
    data=kpi_df,
    x='Season',
    y='Avg_Profit',
    ax=axes[1]
)
axes[1].set_title('Average Profit')
axes[1].set_ylabel('INR')


# Water efficiency
sns.barplot(
    data=kpi_df,
    x='Season',
    y='Avg_Water_Efficiency',
    ax=axes[2]
)
axes[2].set_title('Water Efficiency')
axes[2].set_ylabel('Tonnes / 1000 m³')

plt.suptitle(
    'Seasonal Agriculture Performance — Final Dashboard',
    fontsize=18,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

## Final Project Outcome

### 🏆 Best Season
**Kharif**

### 🌾 Best Crop-Season Combination
**Sugarcane — Kharif**

### ⚠️ Lowest Crop-Season Combination
**Rice — Zaid**

### 📊 Statistical Findings
- Yield: Not statistically significant across seasons
- Profit: Statistically significant across seasons
- Water efficiency: Statistically significant across seasons

### 💧 Strongest Resource Association
**Water usage showed the strongest observed resource-level relationship
with yield.**

### 🎯 Overall Outcome

The project provides a complete analytical framework for understanding
seasonal agricultural performance and identifying opportunities for
better seasonal planning, resource management and economic assessment.

In [ ]:
import os

print("CSV files in Colab:")
for file in os.listdir('/content'):
    if file.endswith('.csv'):
        print(file)

In [ ]:
# ============================================================
# SAVE FINAL PROJECT OUTPUT FILES
# ============================================================

# Final Seasonal KPI
final_season_kpi_export = final_season_kpi.reset_index()

final_season_kpi_export.to_csv(
    '/content/final_season_kpi.csv',
    index=False
)

# Final Crop-Season Ranking
final_ranking_export = final_ranking.copy()

final_ranking_export.to_csv(
    '/content/final_crop_season_ranking.csv',
    index=False
)

print("✅ Final output files saved successfully!")
print()
print("Files:")
print("1. /content/final_season_kpi.csv")
print("2. /content/final_crop_season_ranking.csv")